# Imports

In [11]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def voting_cross_val_predict(probas: list[np.ndarray], weights: np.ndarray[float]) -> np.ndarray[float]:
    weights /= weights.sum()
    return np.sum([proba * weight for proba, weight in zip(probas, weights)], axis=0)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking.parquet')

In [5]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000


## Creating Stacking Database

In [7]:
X_train['voting_proba'] = voting_cross_val_predict(
    [
        X_train['lgbm_proba'],
        X_train['cat_proba'],
        X_train['xgb_proba'],
        X_train['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [8]:
X_test['voting_proba'] = voting_cross_val_predict(
    [
        X_test['lgbm_proba'],
        X_test['cat_proba'],
        X_test['xgb_proba'],
        X_test['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [9]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416


# Machine Learning

In [10]:
features_to_use = ['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba', 'voting_proba']

In [12]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

features_to_use = ['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba', 'voting_proba']

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train[features_to_use], y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 12:36:14,276] A new study created in memory with name: no-name-1871f4b7-77a8-49f9-8af4-314fc51c8a59
Best trial: 10. Best value: 0.949267:   0%|          | 1/500 [00:15<2:13:00, 15.99s/it]

[I 2026-05-18 12:36:30,267] Trial 10 finished with value: 0.9492674605293843 and parameters: {'n_estimators': 81, 'learning_rate': 0.017477914782857356, 'max_depth': 3, 'num_leaves': 9, 'min_child_samples': 170, 'subsample': 0.6715762810423642, 'colsample_bytree': 0.735439410829866, 'reg_alpha': 0.009586648702649374, 'reg_lambda': 0.227010645387839}. Best is trial 10 with value: 0.9492674605293843.


Best trial: 10. Best value: 0.949267:   0%|          | 2/500 [00:17<1:00:03,  7.24s/it]

[I 2026-05-18 12:36:31,362] Trial 1 finished with value: 0.9482886381760309 and parameters: {'n_estimators': 82, 'learning_rate': 0.010199638864485902, 'max_depth': 3, 'num_leaves': 16, 'min_child_samples': 117, 'subsample': 0.6617379785915019, 'colsample_bytree': 0.9961159985467192, 'reg_alpha': 29.62345578793132, 'reg_lambda': 9.777957786427615}. Best is trial 10 with value: 0.9492674605293843.


Best trial: 10. Best value: 0.949267:   1%|          | 3/500 [00:19<40:04,  4.84s/it]  

...

[I 2026-05-18 13:09:37,821] Trial 496 finished with value: 0.9498821998748703 and parameters: {'n_estimators': 270, 'learning_rate': 0.043231015721234965, 'max_depth': 4, 'num_leaves': 13, 'min_child_samples': 228, 'subsample': 0.7966101450322209, 'colsample_bytree': 0.6396987054225923, 'reg_alpha': 7.129919492247167e-05, 'reg_lambda': 0.4917626381574796}. Best is trial 402 with value: 0.9498857992045373.


Best trial: 402. Best value: 0.949886: 100%|█████████▉| 498/500 [33:25<00:04,  2.33s/it]

[I 2026-05-18 13:09:39,391] Trial 497 finished with value: 0.949877802806091 and parameters: {'n_estimators': 271, 'learning_rate': 0.042657691700536904, 'max_depth': 4, 'num_leaves': 12, 'min_child_samples': 284, 'subsample': 0.6627648003785579, 'colsample_bytree': 0.6380435057070192, 'reg_alpha': 9.465356078077375e-05, 'reg_lambda': 1.9709378122005294}. Best is trial 402 with value: 0.9498857992045373.


Best trial: 402. Best value: 0.949886: 100%|██████████| 500/500 [33:25<00:00,  4.01s/it]

[I 2026-05-18 13:09:40,156] Trial 499 finished with value: 0.9498773869093011 and parameters: {'n_estimators': 272, 'learning_rate': 0.0428186246967835, 'max_depth': 4, 'num_leaves': 13, 'min_child_samples': 228, 'subsample': 0.6636462470335653, 'colsample_bytree': 0.6387952995802296, 'reg_alpha': 1.077338411528663e-05, 'reg_lambda': 0.46468879426367793}. Best is trial 402 with value: 0.9498857992045373.
[I 2026-05-18 13:09:40,181] Trial 498 finished with value: 0.9498830261667685 and parameters: {'n_estimators': 272, 'learning_rate': 0.04203852396521302, 'max_depth': 4, 'num_leaves': 13, 'min_child_samples': 233, 'subsample': 0.6784121688169144, 'colsample_bytree': 0.6387267050849619, 'reg_alpha': 1.0246328020384085e-05, 'reg_lambda': 0.4468811257355107}. Best is trial 402 with value: 0.9498857992045373.
Best trial score:
0.9498857992045373

Best params:
{'n_estimators': 279, 'learning_rate': 0.0460336674084307, 'max_depth': 4, 'num_leaves': 14, 'min_child_samples': 231, 'subsample': 

In [13]:
stacking_lg = LGBMClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001593 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

# Submission

In [14]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [15]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [16]:
submission.to_csv('../data/submission/submission.csv', index=False)